<a href="https://colab.research.google.com/github/haze25102583/CNN/blob/main/day5_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [section 1] RNN과 시퀀스 데이터

# [1-1] 3차원 RNN 입력 만들기

### 시퀀스 데이터 및 입력 구조


순서 보존 및 앞 뒤 관계 중요

시계열, 텍스트, 이벤트 로그, 음성 / 영상

Input Shape : [N, T, F] (Samples, Timesteps, Features)

In [ ]:
import numpy as np

values = np.array([[10, 12, 15, 14], [15, 12, 10, 11]], dtype=np.float32)

X = values[..., np.newaxis]                     # 2차원 배열에 feature 축 추가
expected = (2, 4, 1)

print("values: ", values.shape)                                                   # values:  (2, 4)
print("RNN X: ", X.shape)                                                         # RNN X:  (2, 4, 1)

assert X.shape == expected                      # # N·T·F를 자동 검사
print("shape check: PASS")                                                        # shape check: PASS

values:  (2, 4)
RNN X:  (2, 4, 1)
shape check: PASS


# [1-2] 작은 시계열을 Window로 자르기

### 데이터 전처리 (Preprocessing)

 전처리 파이프 라인

        1. 시간 순 분리(Split)
        2. Train 기준 fit
        3. 같은 scaler로 Train/Val/Test 각각 transform
        4. 각 구간(split) 내부에서 별도의 윈도우(Windowing) 생성

Time Split : 시간 순서 분할 (Shuffle x)

Scaling : Train 기준 fit, No Leakage (데이터 누수)

        데이터 누수: 학습(Train)하는 동안, 알 수 없는 '미래 시점(Validation/Test)의 통계적 정보'가 훈련 과정에 섞여 들어가는 것.

        transform 할 때도 Train의 공통 기준 (미래 구간 xx) 을 적용.

Windowing: 1차원의 흐름을 모델이 처리할 수 있는 3차원 ([N, W, 1]) 입력과 정답 짝으로 만들어내는 물리적 작업

        전체 데이터의 길이=L인 입력 윈도우를 1칸 씩 이동시켜
        과거 구간(x), 바로 다음 시점의 값(y)을 짝지어
        여러개의 학습 샘플 만듦
        **  과거 W개 -> 다음 y 생성 (학습 샘플 수 L-W)  **

        다음 값을 예측하는 문제(next-step)의 경우,
        '정답(y)'이 윈도우 바로 뒤에 존재

        맨 마지막 윈도우는 학습 샘플 불가능

In [ ]:
import numpy as np

series = np.array([10, 12, 15, 14, 18]) # 전체 데이터의 길이=L
W = 2                                   # 윈도우의 크기=W
samples = []

for i in range(len(series) - W):        # 학습 샘플 : L-W
    X = series[i:i+W]                   # 입력데이터 : 현재 위치(i) + 윈도우의 크기(W)
    y = series[i+W]                     # 윈도우가 끝나는 = 모델이 맞춰야 할 정답
    samples.append((X, y))
    print(f"X={X} -> y={y}")

assert len(samples) == len(series) - W
print("sample count: ", len(samples))                                            # sample count:  3

X=[10 12] -> y=15
X=[12 15] -> y=14
X=[15 14] -> y=18
sample count:  3


# [1-3] 시간 순서대로 70·15·15 나누기

In [ ]:
import numpy as np

series = np.arange(20)
n = len(series)
i1, i2 = int(n * .7), int(n * .85)

train = series[:i1]                         # shuffle 없이 배열 슬라이싱 : 과거 → 미래 평가
val = series[i1:i2]
test = series[i2:]

print("lengths:", len(train), len(val), len(test))                               # lengths: 14 3 3
print("bounds :", train[-1], val[0], test[0])                                    # bounds : 13 14 17
assert (len(train), len(val), len(test)) == (14, 3, 3)
assert train[-1] < val[0] < test[0]

lengths: 14 3 3
bounds : 13 14 17


# [1-4] Train 기준으로만 표준화

In [ ]:
import numpy as np

train = np.array([10, 12, 14], dtype=np.float32)
val = np.array([16, 18], dtype=np.float32)

mu, sigma = train.mean(), train.std()
train_z = (train - mu) / sigma
val_z = (val - mu) / sigma

print("mu, sigma: ", round(float(mu), 3),                                         # mu, sigma:  12.0 1.633
    round(float(sigma), 3))
print("train mean: ", round(float(train_z.mean()), 6))                            # train mean:  0.0
print("val mean  : ", round(float(val_z.mean()), 3))                          # val mean  :  3.062

assert np.isclose(train_z.mean(), 0.0)
assert not np.isclose(val_z.mean(), 0.0)    # 미래 통계로 다시 계산x

mu, sigma:  12.0 1.633
train mean:  0.0
val mean  :  3.062


# [section 2] RNN 기본 구조

# Hidden State: 과거 정보 요약, 갱신 (ht = f(xt, ht-1))

시퀀스 데이터는 순서를 보존,

3차원 입력 구조 ([N, T, F])

        RNN (순환 신경망) 모델이 데이터의 '순서, 앞 뒤 관계' 학습




시간과 문맥의 위치에 셀(Cell)을 공유하고,

이전 정보를 'Hidden State(은닉 상태)'로 전달

        앞에서 무엇을 보았는 지



# [2-1] hidden state를 두 시점 계산

RNN Cell과 Hidden State (기억의 갱신과 요약)

        현재 시점의 입력 데이터(x t) + 이전 과거에서 전달받은 정보인 은닉 상태(h t−1
        ​) -> 새로운 상태(h t​)

        -> 입력값과 이전 기억에 각각 가중치를 곱해 더함

        -> 활성화 함수(tanh): 정보를 제한된 크기의 벡터로 '요약'하여 다음 시점에 전달

Unroll(시간축 펼침)과 가중치 공유 (Shared Weights)

        Unroll(시간축 펼침):  동일한 가중치(Wx,Wh,b)를 가진 '단 1개의 Cell'을
        데이터의 시점(T) 길이만큼 반복해서 실행

        입력 시퀀스 윈도우의 길이(T)가 길어져도
        모델이 학습해야 할 파라미터(가중치)의 개수는 증가x

In [ ]:
import numpy as np

Wx, Wh, h = 1.0, 0.5, 0.0
sequence = [0.5, -0.2]              # x
states = []                         # y -> h t+1

for t, x in enumerate(sequence, start=1):
    h = np.tanh(x*Wx + h*Wh)
    states.append(h)
    print(f"h{t} = {h:.3f}")

expected = [0.462117, 0.031049]
assert np.allclose(states, expected, atol=1e-3)
print("state check: PASS")

h1 = 0.462
h2 = 0.031
state check: PASS


# [2-2] return_sequences 출력 shape 비교

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# 입력 데이터
# 샘플 2개 × 시점 4개 × 각 시점의 특성 1개
toy = tf.zeros((2, 4, 1))

# 모든 시점을 처리한 후 마지막 상태 h4만 반환
last = layers.SimpleRNN(
    units=3,                # 2층
    return_sequences=False  # 마지막 상태만
)(toy)

# 각 시점의 상태: h1, h2, h3, h4 모두 반환
all_steps = layers.SimpleRNN(
    units=3, return_sequences=True              # T축 남김
)(toy)

print("입력 shape: ", toy.shape)                # 입력 shape:  (2, 4, 1)
print("마지막 상태만: ", last.shape)            # 마지막 상태만:  (2, 3)
print("모든 시점의 상태: ", all_steps.shape)    # 모든 시점의 상태:  (2, 4, 3)

assert last.shape == (2,3)
assert all_steps.shape == (2,4,3)

print("shape check: PASS")

입력 shape:  (2, 4, 1)
마지막 상태만:  (2, 3)
모든 시점의 상태:  (2, 4, 3)
shape check: PASS


# [2-3] SimpleRNN 파라미터 수 확인

RNN은 Shared Weights 이고, 반복 실행되는 단일 셀(Cell) 구조.

파라미터 공식:
        U (F+U+1)

        Wx(입력 가중치): 현재 시점의 입력 특성 수(F) -> 은닉 유닛 수(U) = F x U

        Wh(순환 가중치): 과거 정보를 요약한 시점의 은닉 유닉(U) -> 새로운 은닉 유닛(U) = U x U

        bias, b(편향): 은닉 유닛(U)마다 1씩 부여되는 편향의 개수 = U x 1

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(4,1)),          # T=4, F=1
    layers.SimpleRNN(3, name="rnn"),    # units(U)=3
                                        # RNN => U(FxU+1) = 3x(1+3+1)=15
    layers.Dense(1, name="out")         # Dense: 3x1 + b = 4
])
expected = 3*(1+3+1) + (3*1 + 1)

model.summary()
print("expected: ", expected)               # expected:  19
print("actual  : ", model.count_params())   # actual  :  19

assert model.count_params() == expected
# SimpleRNN(15) + Dense(4) = 19(TotalParams)      파라미터 수를 늘리지 x

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rnn (SimpleRNN)                 │ (None, 3)              │            15 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (Dense)                     │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19 (76.00 B)

 Trainable params: 19 (76.00 B)

 Non-trainable params: 0 (0.00 B)

expected:  19
actual  :  19


# [2-4] padding, masking 확인

서로 다른 시퀀스를 모델에 한 번에 주입하기 위해

시퀀스의 빈자리를 0 같은 PAD로 채움

Masking

        모델에게 해당 위치가 무의미한 PAD 값임을 알려주어,
        RNN이 Hidden State를 갱신 / Loss 계산 시, PAD를 무시

In [2]:
# Masking: 값 0은 hidden state를 오염 x

import tensorflow as tf
from tensorflow.keras import layers

x = tf.constant([[[1.], [2.], [0.], [0.]],
                 [[1.], [2.], [3.], [4.]]])
masking = layers.Masking(mask_value=0.0)    # layers.Masking: not processing garbage data
                                            # 0.0 -> PAD
masked_x = masking(x)
mask = masking.compute_mask(x)
rnn = layers.SimpleRNN(3)
masked_out = rnn(masked_x)
short_out = rnn(x[:1, :2, :])       # 가짜 데이터 0 제외
same = tf.reduce_all(tf.abs(masked_out[:1] - short_out) < 1e-6)     # tf.reduce_all: True, False 연산
print("mask  : ", mask.numpy())     # mask  :  [[ True  True False False]
                                    #           [ True  True  True  True]]
print("output: ", masked_out.shape, bool(same.numpy()))     # output:  (2, 3) True

mask  :  [[ True  True False False]
 [ True  True  True  True]]
output:  (2, 3) True
